# Caso 1: SIMPLE - Calidad de Datos en Retail

---

## Contexto del Negocio

### Descripción del Problema
**MegaStore**, una tienda online de productos electrónicos, ha notado inconsistencias en sus reportes de ventas mensuales. Los gerentes reportan cifras diferentes dependiendo de qué sistema consulten, y el equipo de marketing no puede segmentar correctamente a sus clientes.

### Objetivo Analítico
Evaluar y mejorar la calidad de los datos transaccionales de ventas para:
- Generar reportes confiables de ingresos
- Segmentar clientes de manera efectiva
- Identificar productos con mejor desempeño

### Impacto de la Mala Calidad de Datos
- **Financiero**: Reportes de ingresos incorrectos pueden llevar a decisiones de inversión erróneas
- **Operativo**: Inventario mal calculado por datos duplicados o inconsistentes
- **Estratégico**: Campañas de marketing dirigidas a segmentos incorrectos

---

## Dimensiones de Calidad a Evaluar

En este caso trabajaremos con:

1. **Completitud**: ¿Tenemos todos los datos necesarios?
2. **Exactitud**: ¿Los valores son correctos?
3. **Consistencia**: ¿Los datos son coherentes entre sí?
4. **Integridad**: ¿Se mantienen las relaciones entre tablas?
5. **Razonabilidad**: ¿Los valores están dentro de rangos esperados?
6. **Oportunidad**: ¿Los datos están actualizados?
7. **Unicidad**: ¿Existen registros duplicados?
8. **Validez**: ¿Los formatos son correctos?

---

---

# SOLUCIÓN NIVEL INTERMEDIO

## Objetivo
Recuperar datos cuando sea posible usando:
- Imputación inteligente de valores faltantes
- Corrección de errores comunes
- Detección de outliers con contexto
- Validaciones basadas en reglas de negocio

### Filosofía:
"No eliminar sin antes intentar recuperar información valiosa"

In [1]:
# Instalación de librerías necesarias
# !pip install pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

In [7]:
df = pd.read_parquet("dataset_calidad.parquet")

In [6]:
# Definir relaciones esperadas
categoria_esperada = {
    'Laptop': 'Computadoras',
    'Monitor': 'Computadoras',
    'Mouse': 'Accesorios',
    'Teclado': 'Accesorios',
    'Audífonos': 'Accesorios',
    'Webcam': 'Accesorios',
    'SSD': 'Componentes',
    'RAM': 'Componentes'
}


# Duplicados exactos (excluyendo ID)
cols_duplicados = [c for c in df.columns if c != 'id_transaccion']

In [8]:
# Crear copia para solución intermedia
df_intermedio = df.copy()
print(f"Registros iniciales: {len(df_intermedio)}\n")

# ============================================
# 1. IMPUTACIÓN DE VALORES FALTANTES
# ============================================

# Método de pago: usar la moda (más frecuente)
metodo_pago_moda = df_intermedio['metodo_pago'].mode()[0]
nulos_metodo = df_intermedio['metodo_pago'].isnull().sum()
df_intermedio['metodo_pago'].fillna(metodo_pago_moda, inplace=True)
print(f"- Método de pago: {nulos_metodo} valores imputados con '{metodo_pago_moda}'")

# Región: usar la moda
region_moda = df_intermedio['region'].mode()[0]
nulos_region = df_intermedio['region'].isnull().sum()
df_intermedio['region'].fillna(region_moda, inplace=True)
print(f"- Región: {nulos_region} valores imputados con '{region_moda}'")

# Email: generar email genérico basado en cliente_id
nulos_email = df_intermedio['cliente_email'].isnull().sum()
df_intermedio.loc[df_intermedio['cliente_email'].isnull(), 'cliente_email'] = \
    df_intermedio.loc[df_intermedio['cliente_email'].isnull(), 'cliente_id'].apply(
        lambda x: f'cliente{x}@email.com'
    )
print(f"- Email: {nulos_email} valores generados basados en cliente_id")

# ============================================
# 2. CORRECCIÓN DE VALORES INCORRECTOS
# ============================================
print("\nCORRECCIÓN DE VALORES INCORRECTOS")

# Precios negativos: usar la mediana del producto
precios_neg = df_intermedio['precio_unitario'] <= 0
for producto in df_intermedio[precios_neg]['producto'].unique():
    mediana_precio = df_intermedio[
        (df_intermedio['producto'] == producto) & 
        (df_intermedio['precio_unitario'] > 0)
    ]['precio_unitario'].median()
    
    if pd.notna(mediana_precio):
        df_intermedio.loc[
            (df_intermedio['producto'] == producto) & precios_neg, 
            'precio_unitario'
        ] = mediana_precio

print(f"- Precios negativos/cero corregidos: {precios_neg.sum()} usando mediana por producto")

# Descuentos > 100%: limitar a 0.5 (50%)
descuentos_inv = df_intermedio['descuento'] > 1
df_intermedio.loc[descuentos_inv, 'descuento'] = 0.5
print(f"- Descuentos inválidos corregidos: {descuentos_inv.sum()} limitados a 50%")

# Cantidades extremas: usar mediana del producto
cantidades_ext = (df_intermedio['cantidad'] <= 0) | (df_intermedio['cantidad'] > 100)
for producto in df_intermedio[cantidades_ext]['producto'].unique():
    mediana_cant = df_intermedio[
        (df_intermedio['producto'] == producto) & 
        (df_intermedio['cantidad'] > 0) & 
        (df_intermedio['cantidad'] <= 100)
    ]['cantidad'].median()
    
    if pd.notna(mediana_cant):
        df_intermedio.loc[
            (df_intermedio['producto'] == producto) & cantidades_ext, 
            'cantidad'
        ] = mediana_cant

print(f"- Cantidades extremas corregidas: {cantidades_ext.sum()} usando mediana por producto")

# ============================================
# 3. ESTANDARIZACIÓN DE FORMATOS
# ============================================
print("\nESTANDARIZACIÓN DE FORMATOS")

# Estados de orden
mapeo_estados = {
    'completada': 'Completada',
    'completa': 'Completada',
    'pendiente': 'Pendiente',
    'cancelada': 'Cancelada',
    'cancelado': 'Cancelada',
    'en proceso': 'En Proceso'
}

df_intermedio['estado_orden'] = df_intermedio['estado_orden'].str.lower().str.strip()
df_intermedio['estado_orden'] = df_intermedio['estado_orden'].replace(mapeo_estados)
df_intermedio['estado_orden'] = df_intermedio['estado_orden'].str.title()
print(f"- Estados de orden estandarizados")

# Emails: convertir a minúsculas y limpiar espacios
df_intermedio['cliente_email'] = df_intermedio['cliente_email'].str.lower().str.strip()
print(f"- Emails normalizados a minúsculas")

# ============================================
# 4. CORRECCIÓN DE INCONSISTENCIAS
# ============================================
print("\nCORRECCIÓN DE INCONSISTENCIAS")

# Corregir categorías basadas en producto
df_intermedio['categoria'] = df_intermedio['producto'].map(categoria_esperada)
print(f"- Categorías corregidas según producto")

# Recalcular monto total
df_intermedio['monto_total'] = df_intermedio['cantidad'] * df_intermedio['precio_unitario'] * \
                                (1 - df_intermedio['descuento'])
print(f"- Montos totales recalculados")

# ============================================
# 5. MANEJO DE DUPLICADOS
# ============================================
print("\nMANEJO DE DUPLICADOS")

# Duplicados exactos: mantener el primero
antes_dup = len(df_intermedio)
df_intermedio = df_intermedio.drop_duplicates(subset=cols_duplicados, keep='first')
print(f"- Duplicados exactos eliminados: {antes_dup - len(df_intermedio)}")

# ============================================
# 6. FILTRADO FINAL DE OUTLIERS EXTREMOS
# ============================================
print("\nFILTRADO DE OUTLIERS EXTREMOS")

# Usar IQR para detectar outliers en precio
Q1 = df_intermedio['precio_unitario'].quantile(0.25)
Q3 = df_intermedio['precio_unitario'].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 3 * IQR  # Usar 3*IQR para ser más permisivos

antes_outliers = len(df_intermedio)
df_intermedio = df_intermedio[df_intermedio['precio_unitario'] <= limite_superior]
print(f"- Outliers extremos en precio eliminados: {antes_outliers - len(df_intermedio)}")
print(f"- Límite superior aplicado: ${limite_superior:.2f}")

# Fechas: filtrar solo registros de últimos 6 meses
fecha_limite = datetime.now() - timedelta(days=180)
antes_fechas = len(df_intermedio)
df_intermedio = df_intermedio[
    (df_intermedio['fecha'] >= fecha_limite) & 
    (df_intermedio['fecha'] <= datetime.now())
]
print(f"- Fechas fuera de rango eliminadas: {antes_fechas - len(df_intermedio)}")


print(f"Registros finales: {len(df_intermedio)}")
print(f"Pérdida total: {len(df) - len(df_intermedio)} registros ({((len(df) - len(df_intermedio))/len(df)*100):.1f}%)")


Registros iniciales: 1000025

- Método de pago: 10 valores imputados con 'Transferencia'
- Región: 10 valores imputados con 'Sur'
- Email: 10 valores generados basados en cliente_id

CORRECCIÓN DE VALORES INCORRECTOS
- Precios negativos/cero corregidos: 8 usando mediana por producto
- Descuentos inválidos corregidos: 5 limitados a 50%
- Cantidades extremas corregidas: 6 usando mediana por producto

ESTANDARIZACIÓN DE FORMATOS
- Estados de orden estandarizados
- Emails normalizados a minúsculas

CORRECCIÓN DE INCONSISTENCIAS
- Categorías corregidas según producto
- Montos totales recalculados

MANEJO DE DUPLICADOS
- Duplicados exactos eliminados: 15

FILTRADO DE OUTLIERS EXTREMOS
- Outliers extremos en precio eliminados: 4
- Límite superior aplicado: $3363.46
- Fechas fuera de rango eliminadas: 509194
Registros finales: 490812
Pérdida total: 509213 registros (50.9%)


In [386]:
# Verificación post-limpieza
print("\nVERIFICACIÓN POST-LIMPIEZA (SOLUCIÓN INTERMEDIA)\n")
print(f"- Valores nulos: {df_intermedio.isnull().sum().sum()}")
print(f"- Duplicados exactos: {df_intermedio.duplicated(subset=cols_duplicados).sum()}")
print(f"- Precios negativos: {len(df_intermedio[df_intermedio['precio_unitario'] <= 0])}")
print(f"- Descuentos > 100%: {len(df_intermedio[df_intermedio['descuento'] > 1])}")
print(f"- Cantidades <= 0: {len(df_intermedio[df_intermedio['cantidad'] <= 0])}")
print(f"- Estados únicos: {df_intermedio['estado_orden'].unique()}")


VERIFICACIÓN POST-LIMPIEZA (SOLUCIÓN INTERMEDIA)

- Valores nulos: 20
- Duplicados exactos: 0
- Precios negativos: 0
- Descuentos > 100%: 0
- Cantidades <= 0: 0
- Estados únicos: <ArrowStringArray>
['En Proceso', 'Completada', 'Cancelada', 'Pendiente']
Length: 4, dtype: str


In [9]:
# Verificación post-limpieza
print("\nVERIFICACIÓN POST-LIMPIEZA (SOLUCIÓN INTERMEDIA)\n")
print(f"- Valores nulos: {df_intermedio.isnull().sum().sum()}")
print(f"- Duplicados exactos: {df_intermedio.duplicated(subset=cols_duplicados).sum()}")
print(f"- Precios negativos: {len(df_intermedio[df_intermedio['precio_unitario'] <= 0])}")
print(f"- Descuentos > 100%: {len(df_intermedio[df_intermedio['descuento'] > 1])}")
print(f"- Cantidades <= 0: {len(df_intermedio[df_intermedio['cantidad'] <= 0])}")
print(f"- Estados únicos: {df_intermedio['estado_orden'].unique()}")


VERIFICACIÓN POST-LIMPIEZA (SOLUCIÓN INTERMEDIA)

- Valores nulos: 0
- Duplicados exactos: 0
- Precios negativos: 0
- Descuentos > 100%: 0
- Cantidades <= 0: 0
- Estados únicos: ['Pendiente' 'Completada' 'Cancelada' 'En Proceso']


In [10]:
df_intermedio.to_parquet("intermedio")

### Conclusiones de la Solución Intermedia

**Ventajas:**
- Recupera información valiosa mediante imputación
- Menor pérdida de datos (~15-25%)
- Aplica reglas de negocio
- Trata inconsistencias de formato

**Desventajas:**
- Requiere conocimiento del dominio
- La imputación puede introducir sesgos
- No automatiza validaciones complejas